# PRISMATIC Workshop: Initializing FATES from NEON Remote Sensing

This notebook walks through the PRISMATIC pipeline to use plot-based and remote sensing forest data to generate initial conditions for the FATES (Functionally Assembled Terrestrial Ecosystem Simulator) vegetation model. The main steps in the workflow are:

1) Download and clean NEON remote sensing and forest inventory data
2) Develop a model to estimate vegetation size classes (height of canopy layers) calibrated by known tree sizes in forest inventory plots
3) Develop a model to classify plant functional type (PFT) calibrated on known functional types in forest inventory plots
4) Estimate size classes and PFTs across remote sensing extent to generate FATES initial conditions

**Site:** TEAK (Lower Teakettle), California  
**Year:** 2021  
**Model target:** FATES 

---

## Cyverse Container
1) Sign up for CyVerse account: https://user.cyverse.org/signup
2) Enroll in workshop: https://planion.events/e/esa/am2026/sessions/861641/
3) Wait for approval
4) Go to Discovery Environment: https://de.cyverse.org/dashboard
5) Search "PRISMATIC Tutorial"
6) Launch app

## Motivation

Background on this project and why it's important: https://canva.link/0e9gdg4lr18gdu9

## Setup

In [ ]:
import sys, os
sys.path.insert(0, '..')

from hydra import initialize, compose
from omegaconf import OmegaConf

from utils.utils import build_cache_site, force_rerun
from initialize.inventory import download_veg_structure_data, download_trait_table, prep_veg_structure
from initialize.plots import download_polygons, prep_polygons
from initialize.lidar import download_lidar, download_aop_bbox, normalize_laz, clip_lidar_by_plots
from initialize.lad import prep_lad
from initialize.biomass import prep_biomass
from initialize.hyperspectral import (download_hyperspectral, correct_flightlines,
                                      prep_manual_training_data, prep_aop_imagery,
                                      extract_spectra_from_polygon, train_pft_classifier)
from initialize.generate_initial_conditions import generate_initial_conditions

with initialize(config_path='conf', version_base=None):
    cfg = compose(config_name='config')

site = 'TEAK'
year = '2021'
site_cfg = cfg.sites.run[site][year]
year_aop = site_cfg.year_aop

print(OmegaConf.to_yaml(site_cfg))

# --- Values from the Hydra config (the same ones main.py pulls out) ---
data_raw_aop_path = cfg.paths.data_raw_aop_path
data_raw_inv_path = cfg.paths.data_raw_inv_path
data_int_path     = cfg.paths.data_int_path
data_final_path   = cfg.paths.data_final_path

ic_type          = cfg.others.ic_type
hs_type          = cfg.others.hs_type
month_window     = cfg.others.month_window
n_plots          = cfg.others.n_plots
plot_length      = cfg.others.plot_length
ntree            = cfg.others.ntree
min_distance     = cfg.others.min_distance
use_tiles_w_veg  = cfg.others.use_tiles_w_veg
randomMinSamples = cfg.others.randomMinSamples
aggregate_from_1m_to_2m_res = cfg.others.aggregate_from_1m_to_2m_res
independentValidationSet    = cfg.others.independentValidationSet
pcaInsteadOfWavelengths     = cfg.others.pcaInsteadOfWavelengths
multisite        = cfg.others.multisite
coords_bbox      = cfg.others.coords_bbox
neon_trait_link  = cfg.others.neon_trait_table.neon_trait_link

use_case = "train"

# step(): run a pipeline function only if its outputs aren't already on disk.
# This mirrors main.py's force_rerun/cache wiring, so re-running a cell skips work
# that is already done and returns the existing output path(s) instead.
rerun_status = {k: bool(v) for k, v in site_cfg.force_rerun.items()}

def step(fn, **kwargs):
    cache = build_cache_site(site=site, year_inventory=year, year_aop=year_aop,
                             data_raw_aop_path=data_raw_aop_path,
                             data_raw_inv_path=data_raw_inv_path,
                             data_int_path=data_int_path,
                             hs_type=hs_type, coords_bbox=coords_bbox)
    return force_rerun(cache, force=rerun_status)(fn)(**kwargs)

---

## 1. Download and clean data

Download and clean NEON remote sensing and forest inventory data

<img src="docs/1_workflow.png" width="90%">

In [ ]:
# Download stem-level vegetation structure data (species, DBH, height, location)
veg_structure_path, sampling_effort_path = step(
    download_veg_structure_data, site=site, data_path=data_raw_inv_path)

# Download NEON plot boundary polygons
neon_plots_path = step(download_polygons, data_path=data_raw_inv_path)

# Download the NEON trait table (used later for allometric biomass equations)
trait_table_path = step(
    download_trait_table, download_link=neon_trait_link, data_path=data_raw_inv_path)

# Download AOP LiDAR (.laz) + hyperspectral imagery for the site's flight year.
# If coords_bbox is set in conf, both are fetched clipped to that box in one call instead.
if not coords_bbox:
    laz_path, tif_path = step(
        download_lidar, site=site, year=year_aop,
        lidar_path=data_raw_aop_path, use_tiles_w_veg=use_tiles_w_veg)
    hs_path = step(
        download_hyperspectral, site=site, year=year_aop,
        data_raw_aop_path=data_raw_aop_path, hs_type=hs_type)
else:
    hs_path, laz_path, tif_path = step(
        download_aop_bbox, site=site, year=year_aop,
        path=data_raw_aop_path, hs_type=hs_type, coords_bbox=coords_bbox)

### 1b. Clean forest inventory data

**Functions:** `prep_veg_structure`, `prep_polygons`

Clean and filter stem measurements to the target year, assign PFT labels to individual plants, and partition NEON plots into spatial units suitable for linking to remote sensing pixels.

In [ ]:
# Filter stems to the target year window and assign PFT labels
inventory_file_path, sampling_effort_path = step(
    prep_veg_structure, site=site, year_inv=year, year_aop=year_aop,
    data_path=data_raw_inv_path, month_window=month_window)

# Partition plot polygons into spatial subunits for linking to remote sensing
partitioned_plots_path = step(
    prep_polygons, input_data_path=neon_plots_path,
    sampling_effort_path=sampling_effort_path, inventory_path=inventory_file_path,
    site=site, year=year, output_data_path=data_int_path)

<img src="docs/1b_TEAK.png" width="100%">

Count of taxonomic types in NEON forest inventory plots at TEAK in 2021


### 2a. Processing LiDAR: Normalization and Clipping

**Functions:** `normalize_laz`, `clip_lidar_by_plots`

Height-normalize lidar point clouds (removing terrain so heights reflect canopy, not topography), then clip to inventory plot boundaries to align 3D structure data with field observations.

In [ ]:
# Subtract the digital terrain model so z reflects canopy height above ground
normalized_laz_path = step(
    normalize_laz, laz_path=laz_path, site=site, year=year, output_path=data_int_path)

# Clip normalized point clouds to the partitioned plot boundaries
clipped_laz_path = step(
    clip_lidar_by_plots, laz_path=normalized_laz_path, tif_path=tif_path,
    site_plots_path=partitioned_plots_path, site=site, year=year,
    output_laz_path=data_int_path, end_result=True)

<img src="docs/2a_tilelaz.png" width="45%"> <img src="docs/2a_plotlaz.png" width="20%">

Left, 1 km2 NEON AOP normalized lidar tile point cloud. Right, lidar point cloud clipped to plot extent

### 2b. Deriving Canopy Structure: Leaf Area Density Profiles

**Function:** `prep_lad`

Compute vertical leaf area density (LAD) profiles from normalized lidar within each plot, stratified by size class — a key structural descriptor of canopy layering and vegetation density.

### FATES Cohort Structure

The LAD profiles map directly onto FATES cohort height classes. Each cohort in FATES represents plants of similar height competing for light — the canopy layering captured here defines the initial vertical structure of the simulated forest.

In [ ]:
# Compute leaf area density (LAD) profiles per plot, stratified by PFT size class
prep_lad_path = step(
    prep_lad, laz_path=clipped_laz_path, inventory_path=inventory_file_path,
    site=site, year=year, output_path=data_int_path, use_case=use_case)

<img src="docs/2b_plot_52_321100_4097500_lad.png" width="90%">

Leaf area density plot. Size classes for FARTES cohorts are marked at the local maxima

### 2c. Estimating Biomass

**Function:** `prep_biomass`

Apply allometric equations to stem measurements (using a NEON trait table) to estimate above-ground biomass per plant functional type per plot — providing a carbon stock summary alongside structural data.

In [ ]:
# Estimate above-ground biomass per stem via species-specific allometry (NEON trait table)
biomass_path = step(
    prep_biomass, data_path=inventory_file_path, site_plots_path=partitioned_plots_path,
    sampling_effort_path=sampling_effort_path, site=site, year=year,
    data_int_path=data_int_path, neon_trait_table_path=trait_table_path,
    end_result=False)

<img src="docs/2c_biomassraster.png" width="50%">

Example of biomass map generated across NEON SOAP site (also in California)

<img src="docs/2c_biomassscrnsht.png" width="50%">

In this workflow we also calculate stem density, basal area, and biomass per plot

---

## 3. Develop plant functional type (PFT) classifier

Develop a model to classify plant functional type (PFT) calibrated on known functional types in forest inventory plots

<img src="docs/3_workflow.png" width="90%">

### 3a. Preparing Hyperspectral Imagery

**Functions:** `prep_aop_imagery`

If using hyperspectral flightlines, apply BRDF and topographic corrections. Then stack hyperspectral bands with lidar-derived rasters into a single multi-layer image ready for pixel-level classification.

In [ ]:
# These remote-sensing steps run for every ic_type except "field_inv_plots".
if ic_type != "field_inv_plots":
    # BRDF + topographic correction is only needed for flightline hyperspectral
    if hs_type == "flightline":
        step(correct_flightlines, site=site, year_inv=year, year_aop=year_aop,
             data_raw_aop_path=data_raw_aop_path, data_int_path=data_int_path)

    # Stack hyperspectral bands + LiDAR-derived rasters into one multi-layer image
    stacked_aop_path = step(
        prep_aop_imagery, site=site, year=year, hs_type=hs_type,
        hs_path=hs_path, tif_path=tif_path, data_int_path=data_int_path,
        use_tiles_w_veg=use_tiles_w_veg)

<img src="docs/3a_single_multi_raster.png" width="80%">

Visualization of the raster stack used as training data. 

From 
https://www.neonscience.org/resources/learning-hub/tutorials/dc-multiband-rasters-r

### 3b. Building the Training Dataset

**Functions:** `prep_manual_training_data`, `extract_spectra_from_polygon`

Overlay inventory-derived crown polygons onto the stacked AOP imagery to extract per-pixel spectral signatures, producing a labeled training table (remote sensing features + PFT class) for the classifier.

In [ ]:
if ic_type != "field_inv_plots":
    # Build labelled training crowns from the inventory + biomass
    training_crown_shp_path = step(
        prep_manual_training_data, site=site, year=year,
        data_raw_inv_path=data_raw_inv_path, data_int_path=data_int_path,
        biomass_path=biomass_path)

    # Extract per-pixel spectra within each crown polygon and attach PFT labels
    training_spectra_csv_path = step(
        extract_spectra_from_polygon, site=site, year=year,
        shp_path=training_crown_shp_path, data_int_path=data_int_path,
        data_final_path=data_final_path, stacked_aop_path=stacked_aop_path,
        use_case=use_case, aggregate_from_1m_to_2m_res=aggregate_from_1m_to_2m_res,
        ic_type=ic_type)

<img src="docs/3b_manualcrownssnrnsht.png" width="60%">

Screenshot of a few manually labelled polygons used as training data.

### 3c. Training the PFT Classifier

**Function:** `train_pft_classifier`

Train a Random Forest model on the labeled spectral training data (collapsing dimensionality of hyperspectral data with principle components analysis) and evaluate accuracy — producing a model that can predict PFT identity for any pixel in the scene.

Here we use a pre-trained model on three NEON sites, rather than a single NEON tile that we're working with in this workshop

In [ ]:
# Train the Random Forest PFT classifier on the labelled spectra; report held-out accuracy
sites = [site]
rf_model_path = step(
    train_pft_classifier, sites=sites, data_int_path=data_int_path,
    pcaInsteadOfWavelengths=pcaInsteadOfWavelengths, ntree=ntree,
    randomMinSamples=randomMinSamples, independentValidationSet=independentValidationSet)

<img src="docs/3c_rf_CMnorm.png" width="70%">

Confusion matrix of random forest performance on training data

<img src="docs/3c_rf_FeatImp.png" width="70%">

Rank of raster layers (or features) used in training in order of importance

<img src="docs/3c_uncertainty_agreement_hist.png" width="70%">

Per pixel agreement through k-fold cross-validation

<img src="docs/3c_52_321100_4097500_comparison.png" width="90%">

<img src="docs/3c_54_321300_4097500_comparison.png" width="90%">

Two plots where we compare RGB, CHM, and classified PFT rasters.

---

## 4. Generate FATES intial conditions

Estimate size classes and PFTs across remote sensing extent to generate FATES initial conditions


**Function:** `generate_initial_conditions`

Apply the trained classifier wall-to-wall across the site, then aggregate pixel-level PFT predictions and biomass into FATES cohort and patch files — the end product that initializes a land surface model with spatially-informed vegetation structure. The initial conditions for fates are a two tab-delimited files, one for the cohort information within each patch and one for the patch information within the larger site.

<img src="docs/4_workflow.png" width="90%">

In [ ]:
# Classify wall-to-wall, aggregate to cohorts/patches, and write FATES IC files
use_case = "predict"
ic_type_path = os.path.join(data_final_path, site, year, ic_type)
os.makedirs(ic_type_path, exist_ok=True)

cohort_path, patch_path = step(
    generate_initial_conditions, site=site, year_inv=year, year_aop=year_aop,
    data_raw_aop_path=data_raw_aop_path, data_int_path=data_int_path,
    data_final_path=data_final_path, rf_model_path=rf_model_path,
    stacked_aop_path=os.path.join(data_int_path, site, year, 'stacked_aop'),
    biomass_path=os.path.join(data_int_path, site, year, 'biomass/pp_veg_structure_IND_IBA_IAGB_live.csv'),
    use_case=use_case, ic_type=ic_type, ic_type_path=ic_type_path,
    n_plots=n_plots, min_distance=min_distance, plot_length=plot_length,
    aggregate_from_1m_to_2m_res=aggregate_from_1m_to_2m_res,
    pcaInsteadOfWavelengths=pcaInsteadOfWavelengths, multisite=multisite)

print('cohort file:', cohort_path)
print('patch file:', patch_path)

<img src="docs/4_cohortfile.png" width="100%">

Cohort file example

<img src="docs/4_patchfile.png" width="90%">

Patch file example

<img src="docs/4_rs_wall2wall_agb.png" width="50%"> <img src="docs/4_rs_wall2wall_ba.png" width="50%"> <img src="docs/4_rs_wall2wall_lai.png" width="50%"> <img src="docs/4_rs_wall2wall_leafbiom.png" width="50%"> <img src="docs/4_rs_wall2wall_stemdens.png" width="50%">

Summary community measurements across this TEAK AOP tile  

---

## 5. Where We're Going: FATES Simulations

The initial conditions we just generated feed directly into FATES simulations. Below are example outputs from FATES runs initialized with PRISMATIC data — showing how the spatially-informed PFT structure and size classes translate into simulated forest dynamics over time.

<img src="docs/5_fatesICcomp.png" width="60%">

Comparison initial conditions from field and remote sensing sources.

<img src="docs/5_fatessims.png" width="60%">

Visualization of FATES PFT distributions through time from varying initial conditions sources